<a href="https://colab.research.google.com/github/pratip/llm_engineering/blob/main/week3/exercises/speech_to_text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q --upgrade bitsandbytes accelerate

In [2]:
import os
import requests

from IPython.display import Markdown, display, update_display

from google.colab import userdata
from huggingface_hub import login
from openai import OpenAI
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch

In [3]:
LLAMA_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
AUDIO_FILE_PATH = "/content/sample_data/audio_samples/simple_speech_4.mp3"

In [4]:
# Login to HF.
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# Open the audio file.
# audio_file = open(AUDIO_FILE_PATH, "rb")

In [ ]:
# Open soruce transcription using HF pipeline
from transformers import pipeline

pipe = pipeline(
    task="automatic-speech-recognition",
    model="openai/whisper-large-v3",
    dtype=torch.float16,
    device="cuda",
    return_timestamps=True,
)

result = pipe(
    AUDIO_FILE_PATH,
    language="en",
)
transcription = result["text"]
print(transcription)

In [6]:
system_prompt = """
You are a helpful office assistant who can produce minutes of meetings from
transcripts with summary, key discussion points and action items with owners.
You should use nicely formatted markdown approach without any code blocks.
"""

user_prompt = f"""
Below is an extract of a meeting. A speech is being delivered by the key person.
Please write minutes in markdown without code blocks, including:
- a summary with any attendees dedipherable
- a data time if derivable
- key takeaways
- action items if any

Transcription:
{transcription}
"""

message = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [7]:
# 4‑bit quantization.
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
# Tokenizing and streaming output response.
tokenizer = AutoTokenizer.from_pretrained(LLAMA_MODEL)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(message, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA_MODEL, device_map="auto", quantization_config=quant_config)
outputs = model.generate(inputs, max_new_tokens=2048, streamer=streamer)